# 01 - Analyse exploratoire des donnees

Objectif : comprendre les sources avant de modeliser. Le sujet indique volontairement des incoherences, biais et granularites differentes. Ce notebook ne cherche donc pas a "faire confiance" aux donnees : il les audite.

Decisions attendues :
- distinguer faits observes et hypotheses ;
- identifier les ruptures de granularite pays / destination / review / campagne ;
- documenter les risques avant nettoyage.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1. Chargement multi-source et dimensions

In [ ]:
raw = ROOT / "data" / "raw"

destinations = pd.read_csv(raw / "01_destinations_brut.csv")
market = pd.read_excel(raw / "02_signaux_marche.xlsx")
reviews = pd.read_json(raw / "03_reviews_reseaux.json")
external = pd.read_csv(raw / "04_facteurs_externes.csv", sep=";")
campaigns = pd.read_json(raw / "06_campaign_history.json")

datasets = {
    "destinations": destinations,
    "market": market,
    "reviews": reviews,
    "external": external,
    "campaigns": campaigns,
}

overview = []
for name, df in datasets.items():
    overview.append({
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
    })
pd.DataFrame(overview)

Interpretation : les fichiers n'ont pas la meme granularite. `market` est mensuel au niveau pays, `destinations` est pays-destination, `reviews` est transactionnel, `external` repete des signaux externes, et `campaigns` est un historique court. Une jointure directe sans aggregation creerait des doublons artificiels.

In [ ]:
for name, df in datasets.items():
    print(f"\n{name.upper()} - shape={df.shape}")
    display(pd.DataFrame({"dtype": df.dtypes, "missing": df.isna().sum(), "nunique": df.nunique(dropna=True)}))
    display(df.head(3))

## 2. Statistiques descriptives et distributions numeriques

In [ ]:
numeric_dest = destinations.select_dtypes(include="number")
display(numeric_dest.describe().T)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), numeric_dest.columns):
    sns.histplot(destinations[col], kde=True, ax=ax)
    ax.set_title(f"Distribution - {col}")
plt.tight_layout()

Interpretation : `visitors` a une echelle beaucoup plus large que les scores. Toute combinaison de variables doit donc normaliser les grandeurs pour eviter qu'un volume historique ecrase la qualite ou le sentiment.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, ["attractiveness", "cost", "rating", "visitors"]):
    sns.boxplot(y=destinations[col], ax=ax)
    ax.set_title(f"Boxplot - {col}")
plt.tight_layout()

## 3. Correlations et relations utiles

In [ ]:
corr = numeric_dest.corr(numeric_only=True)
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="vlag", center=0)
plt.title("Correlation des variables destinations")
plt.show()

In [ ]:
sns.pairplot(destinations[["attractiveness", "cost", "rating", "visitors"]], corner=True)
plt.suptitle("Pairplot des variables quantitatives", y=1.02)
plt.show()

Interpretation : les correlations ne doivent pas etre confondues avec des causalites. Elles servent ici a reperer les redondances et les risques de scoring, pas a justifier seules une recommandation business.

## 4. Variables categorielles, cardinalites et incoherences de libelles

In [ ]:
categorical_checks = []
for name, df in datasets.items():
    for col in df.select_dtypes(include="object").columns:
        categorical_checks.append({
            "dataset": name,
            "column": col,
            "nunique": df[col].nunique(dropna=True),
            "examples": sorted(df[col].dropna().astype(str).unique())[:8],
        })
pd.DataFrame(categorical_checks)

In [ ]:
country_values = pd.concat([
    destinations["country"].astype(str),
    market["country"].astype(str),
    reviews["country"].astype(str),
    external["country"].astype(str),
    campaigns["country"].astype(str),
], ignore_index=True)
country_values.value_counts().head(25)

Interpretation : les pays apparaissent avec des casses differentes (`France`, `france`, `FRANCE`). Le nettoyage doit harmoniser la cle, mais cette anomalie doit rester documentee car elle prouve que les sources ne partagent pas de referentiel commun.

## 5. Valeurs manquantes, doublons et anomalies metier

In [ ]:
quality_rows = []
for name, df in datasets.items():
    for col in df.columns:
        quality_rows.append({
            "dataset": name,
            "column": col,
            "missing_rate": df[col].isna().mean(),
            "nunique": df[col].nunique(dropna=True),
        })
quality = pd.DataFrame(quality_rows)
display(quality.sort_values("missing_rate", ascending=False))

plt.figure(figsize=(11, 5))
sns.barplot(data=quality, x="column", y="missing_rate", hue="dataset")
plt.xticks(rotation=45, ha="right")
plt.title("Taux de valeurs manquantes par colonne")
plt.tight_layout()

In [ ]:
campaign_anomalies = campaigns.assign(
    budget_numeric=pd.to_numeric(campaigns["campaign_budget"], errors="coerce"),
    contradiction=lambda d: ((d["status"].str.upper() == "SUCCESS") & (d["roi"] <= 0))
    | ((d["status"].str.upper() == "FAIL") & (d["roi"] > 0))
)
campaign_anomalies[campaign_anomalies["contradiction"] | campaign_anomalies["budget_numeric"].isna()]

Interpretation : un budget `unknown` et des statuts contradictoires avec le ROI ne doivent pas etre corriges arbitrairement. La bonne posture gouvernance est de les flagger, puis de limiter leur poids dans le scoring.

## 6. Relations entre fichiers et couverture des jointures

In [ ]:
def key_frame(df):
    return df.assign(
        country_key=df["country"].astype(str).str.strip().str.title(),
        destination_key=df["destination"].astype(str).str.strip().str.replace("city_", "City_", case=False, regex=False),
    )[["country_key", "destination_key"]].drop_duplicates()

destination_keys = key_frame(destinations)
review_keys = key_frame(reviews)
external_keys = key_frame(external)
campaign_keys = key_frame(campaigns)

coverage = pd.DataFrame([
    {"source": "reviews", "matched_destination_keys": len(destination_keys.merge(review_keys)), "source_keys": len(review_keys)},
    {"source": "external", "matched_destination_keys": len(destination_keys.merge(external_keys)), "source_keys": len(external_keys)},
    {"source": "campaigns", "matched_destination_keys": len(destination_keys.merge(campaign_keys)), "source_keys": len(campaign_keys)},
])
coverage["coverage_rate"] = coverage["matched_destination_keys"] / coverage["source_keys"]
coverage

Conclusion EDA : le projet doit construire une GOLD DATA par aggregation controlee. Les signaux pays ne remplacent pas les signaux destination ; ils enrichissent la recommandation avec une hypothese explicite de potentiel marche.